In [65]:
import os
# #df_conflitos_score
#df_acionamentos_enriquecido_limpo, df_acion_semFaixa, df_acion_semDescricao, df_acion_semOrigem 
df_PAYJOY.to_csv('df_PAYJOY.csv', sep=';', index=False, decimal=',', encoding='utf-8-sig')
os.startfile('df_PAYJOY.csv')  # abre com o programa padrão

In [ ]:
import pandas as pd
from dotenv import load_dotenv
from utils.db_connection import get_connection

conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
sql_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\Daily_PayJoy.sql"

def df_consulta(conn_bd2, sql_file):
    # Lê o conteúdo da query
    with open(sql_file, "r", encoding="utf-8") as f:
        query = f.read()

    cursor = conn_bd2.cursor()
    
    try:
        # Executa a query completa
        cursor.execute(query)
        
        # Percorre todos os resultados até encontrar um com dados
        columns = None
        data = None
        
        # Tenta buscar o primeiro resultado
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            data = cursor.fetchall()
        
        # Avança pelos resultados até encontrar o último com dados
        while cursor.nextset():
            if cursor.description:
                columns = [column[0] for column in cursor.description]
                data = cursor.fetchall()
        
        # Verifica se encontrou dados
        if columns is None or data is None:
            raise ValueError("Nenhum resultado encontrado na query")
        
        # Cria o DataFrame
        df = pd.DataFrame.from_records(data, columns=columns)
        
        return df
        
    except Exception as e:
        print(f"Erro ao executar query: {e}")
        raise
    finally:
        cursor.close()

# Executa a consulta
df_consulta_df = df_consulta(conn_bd2, sql_file)

# Verifica quantas datas únicas existem
datas_unicas = df_consulta_df['DATA'].unique()
print(f"Datas encontradas: {datas_unicas}")
print(f"Total de datas: {len(datas_unicas)}")

# Define automaticamente as colunas de indicadores
# Todas as colunas EXCETO 'DATA' e 'FAIXA'
colunas_todas = df_consulta_df.columns.tolist()
colunas_id = ['DATA', 'FAIXA']  # Colunas identificadoras
value_columns = [col for col in colunas_todas if col not in colunas_id]

print(f"\nColunas identificadoras: {colunas_id}")
print(f"Total de indicadores encontrados: {len(value_columns)}")

# Faz o unpivot (melt) mantendo DATA e FAIXA como identificadores
df_transposto = df_consulta_df.melt(
    id_vars=['DATA', 'FAIXA'],
    value_vars=value_columns,
    var_name='Indicador',
    value_name='Valor'
)

print(f"\nDataFrame após melt:")
print(df_transposto.head(10))

# Pivota para colocar as datas como colunas
df_final = df_transposto.pivot_table(
    index=['FAIXA', 'Indicador'],
    columns='DATA',
    values='Valor',
    aggfunc='first'  # Usa o primeiro valor caso haja duplicatas
).reset_index()

# Remove o nome do índice das colunas (fica mais limpo)
df_final.columns.name = None

# Ordena por FAIXA e Indicador
df_final = df_final.sort_values(['FAIXA', 'Indicador']).reset_index(drop=True)

# Visualiza o resultado
print(f"\n{'='*60}")
print(f"Estrutura final:")
print(f"Total de linhas: {len(df_final)}")
print(f"Faixas únicas: {df_final['FAIXA'].unique()}")
print(f"Colunas: {df_final.columns.tolist()}")
print(f"{'='*60}")
print("\nPrimeiras 30 linhas:")
print(df_final.head(30))

df_final

Datas encontradas: [datetime.date(2025, 12, 4)]
Total de datas: 1

Colunas identificadoras: ['DATA', 'FAIXA']
Total de indicadores encontrados: 23

DataFrame após melt:
         DATA       FAIXA            Indicador  Valor
0  2025-12-04   DPD 16-30   Assigned_Portfolio   1614
1  2025-12-04   DPD 31-60   Assigned_Portfolio   2680
2  2025-12-04   DPD 61-90   Assigned_Portfolio   2084
3  2025-12-04  DPD 91-120   Assigned_Portfolio   1869
4  2025-12-04   DPD 16-30  Reachable_Portfolio      0
5  2025-12-04   DPD 31-60  Reachable_Portfolio      0
6  2025-12-04   DPD 61-90  Reachable_Portfolio      0
7  2025-12-04  DPD 91-120  Reachable_Portfolio      0
8  2025-12-04   DPD 16-30  Contacted_Portfolio   1423
9  2025-12-04   DPD 31-60  Contacted_Portfolio   2369

Estrutura final:
Total de linhas: 92
Faixas únicas: ['DPD 16-30' 'DPD 31-60' 'DPD 61-90' 'DPD 91-120']
Colunas: ['FAIXA', 'Indicador', datetime.date(2025, 12, 4)]

Primeiras 30 linhas:
        FAIXA                                  Indi

,FAIXA,Indicador,2025-12-04
0,DPD 16-30,AHT_Excluding_Short_Calls,2110
1,DPD 16-30,Active_Agents,4
2,DPD 16-30,Assigned_Portfolio,1614
3,DPD 16-30,Contacted_Portfolio,1423
4,DPD 16-30,Debt_Not_Recognized,0
...,...,...,...
87,DPD 91-120,Total_Call_Attempts,3293
88,DPD 91-120,Total_RPCs_Right_Party_Contacts,0
89,DPD 91-120,Unique_Customers_Reached,68
90,DPD 91-120,Unique_Customers_Reached_Excl_Short_Calls,47


In [7]:
import pandas as pd
from dotenv import load_dotenv
from utils.db_connection import get_connection
from datetime import datetime, timedelta
import os
import sys
import re

conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
sql_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\Daily_PayJoy_prod.sql"
xlsx_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\base_PAYJOY.xlsx"

def identificar_maior_data_xlsx(xlsx_path):
    """
    Identifica a maior data presente nas colunas do arquivo Excel.
    Encerra a execução se o arquivo não existir.
    """
    if not os.path.exists(xlsx_path):
        print(f"\n{'='*80}")
        print(f"ERRO CRÍTICO: Arquivo não encontrado!")
        print(f"Caminho: {xlsx_path}")
        print(f"{'='*80}")
        print("\nEncerrando execução...")
        sys.exit(1)
    
    try:
        df_temp = pd.read_excel(xlsx_path, nrows=5)
        print(f"Arquivo Excel lido com sucesso!")
        
        colunas = df_temp.columns.tolist()
        print(f"\nColunas encontradas no arquivo: {colunas}")
        
        ultima_data = None
        
        for col in reversed(colunas):
            if col in ['FAIXA', 'Indicador']:
                continue
            
            try:
                if isinstance(col, (pd.Timestamp, datetime)):
                    if isinstance(col, pd.Timestamp):
                        ultima_data = col.to_pydatetime()
                    else:
                        ultima_data = col
                    print(f"\nMaior data encontrada: {ultima_data.strftime('%Y-%m-%d')} (coluna já era datetime)")
                    return ultima_data
                
                col_limpo = str(col).strip()
                
                for formato in ['%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d', '%d-%m-%Y']:
                    try:
                        data = datetime.strptime(col_limpo, formato)
                        ultima_data = data
                        print(f"\nMaior data encontrada: {ultima_data.strftime('%Y-%m-%d')} (coluna: '{col}')")
                        return ultima_data
                    except:
                        continue
            except Exception as e:
                continue
        
        if ultima_data is None:
            print("\n" + "="*80)
            print("ERRO: Nenhuma data válida encontrada nas colunas do arquivo!")
            print("Colunas analisadas:", [col for col in colunas if col not in ['FAIXA', 'Indicador']])
            print("="*80)
            print("\nEncerrando execução...")
            sys.exit(1)
        
        return ultima_data
            
    except Exception as e:
        print(f"\n{'='*80}")
        print(f"ERRO ao ler arquivo Excel: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        print("\nEncerrando execução...")
        sys.exit(1)

def calcular_data_inicio(maior_data_xlsx):
    """
    Calcula a data de início para a consulta (dia seguinte à última data do arquivo).
    """
    if maior_data_xlsx is None:
        print(f"\n{'='*80}")
        print("ERRO: Não foi possível identificar a data inicial!")
        print(f"{'='*80}")
        sys.exit(1)
    
    data_inicio = maior_data_xlsx.date() + timedelta(days=1)
    print(f"\nData início para consulta: {data_inicio.strftime('%Y-%m-%d')}")
    return data_inicio

def df_consulta(conn_bd2, sql_file, data_inicio):
    """
    Executa a consulta substituindo a variável @dataIni no arquivo SQL.
    
    Parâmetros:
    -----------
    conn_bd2 : pyodbc.Connection
        Conexão com o banco de dados
    sql_file : str
        Caminho do arquivo SQL
    data_inicio : datetime.date
        Data a ser substituída na variável @dataIni
    
    Retorna:
    --------
    pd.DataFrame
        DataFrame com os resultados da consulta
    """
    # Lê o conteúdo da query
    with open(sql_file, "r", encoding="utf-8") as f:
        query_original = f.read()
    
    # Formata a data no formato YYYY-MM-DD
    data_str = data_inicio.strftime('%Y-%m-%d')
    
    print(f"\n[SUBSTITUIÇÃO] Buscando padrão '@dataIni' no SQL...")
    print(f"[SUBSTITUIÇÃO] Formatando data: {data_str}")
    
    # Usa regex para encontrar e substituir de forma robusta
    # Padrão: declare @dataIni as date = 'YYYY-MM-DD' (com variações de espaço)
    padrao = r"declare\s+@dataIni\s+as\s+date\s*=\s*'[^']*'"
    
    # Verificar se encontrou o padrão
    if re.search(padrao, query_original, re.IGNORECASE):
        print(f"[✓] Padrão encontrado no SQL")
        substituicao = f"declare @dataIni as date = '{data_str}'"
        query_modificada = re.sub(padrao, substituicao, query_original, flags=re.IGNORECASE)
        
        # Log das mudanças
        linhas_original = query_original.split('\n')
        linhas_modificada = query_modificada.split('\n')
        
        for i, (orig, mod) in enumerate(zip(linhas_original[:50], linhas_modificada[:50])):
            if orig != mod:
                print(f"\n[LINHA {i+1}] Original:")
                print(f"  {orig.strip()}")
                print(f"[LINHA {i+1}] Modificada:")
                print(f"  {mod.strip()}")
    else:
        print(f"[✗] AVISO: Padrão '@dataIni' NÃO encontrado no SQL!")
        print(f"[✗] A consulta será executada sem modificação da data")
        query_modificada = query_original
    
    cursor = conn_bd2.cursor()
    
    try:
        print(f"\n[EXECUÇÃO] Executando consulta com @dataIni = '{data_str}'...")
        cursor.execute(query_modificada)
        
        # Navega pelos resultados até encontrar o último SELECT
        columns = None
        data = None
        
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            data = cursor.fetchall()
            print(f"[✓] Primeiro result set encontrado: {len(data)} linhas")
        
        result_set_count = 1
        while cursor.nextset():
            if cursor.description:
                result_set_count += 1
                columns = [column[0] for column in cursor.description]
                data = cursor.fetchall()
                print(f"[✓] Result set #{result_set_count} encontrado: {len(data)} linhas")
        
        if columns is None or data is None:
            raise ValueError("Nenhum resultado encontrado na query")
        
        df = pd.DataFrame.from_records(data, columns=columns)
        print(f"\n[✓] Query retornou {len(df)} linhas com {len(columns)} colunas")
        
        return df
        
    except Exception as e:
        print(f"\n[✗] Erro ao executar query: {e}")
        import traceback
        traceback.print_exc()
        raise
    finally:
        cursor.close()

# ============================================================================
# EXECUÇÃO PRINCIPAL
# ============================================================================

print("="*80)
print("INICIANDO PROCESSAMENTO PayJoy")
print("="*80)

# 1. Identifica a maior data no arquivo Excel
maior_data_xlsx = identificar_maior_data_xlsx(xlsx_file)

# 2. Calcula data de início (dia seguinte à última data do arquivo)
data_inicio = calcular_data_inicio(maior_data_xlsx)

# 3. Verifica se há dados para processar
ontem = datetime.now().date() - timedelta(days=1)

if data_inicio > ontem:
    print(f"\n{'='*80}")
    print("Relatório já está atualizado! Nenhuma data para processar.")
    print(f"{'='*80}")
    df_resultado = pd.DataFrame()
else:
    print(f"\nPeríodo a processar: de {data_inicio.strftime('%Y-%m-%d')} até {ontem.strftime('%Y-%m-%d')}")
    
    # 4. Executa a consulta
    df_resultado = df_consulta(conn_bd2, sql_file, data_inicio)
    
    print(f"\n{'='*80}")
    print("CONSULTA EXECUTADA COM SUCESSO!")
    print(f"{'='*80}")
    print(f"Total de linhas retornadas: {len(df_resultado)}")
    print(f"Colunas: {df_resultado.columns.tolist()}")
    print(f"{'='*80}")
    
    if len(df_resultado) > 0:
        print("\nPrimeiras 10 linhas do resultado:")
        print(df_resultado.head(10))
        
        print("\nDatas únicas no resultado:")
        if 'DATA' in df_resultado.columns:
            datas_unicas = sorted(df_resultado['DATA'].unique())
            print(datas_unicas)

conn_bd2.close()
print("\nConexão fechada.")
print(f"\nDataFrame 'df_resultado' disponível com {len(df_resultado)} linhas")

INICIANDO PROCESSAMENTO PayJoy
Arquivo Excel lido com sucesso!

Colunas encontradas no arquivo: ['FAIXA', 'Indicador', datetime.datetime(2026, 1, 2, 0, 0)]

Maior data encontrada: 2026-01-02 (coluna já era datetime)

Data início para consulta: 2026-01-03

Período a processar: de 2026-01-03 até 2026-01-15

[SUBSTITUIÇÃO] Buscando padrão '@dataIni' no SQL...
[SUBSTITUIÇÃO] Formatando data: 2026-01-03
[✓] Padrão encontrado no SQL

[LINHA 9] Original:
  declare @dataIni as date = '2026-01-01';
[LINHA 9] Modificada:
  declare @dataIni as date = '2026-01-03';

[EXECUÇÃO] Executando consulta com @dataIni = '2026-01-03'...
[✓] Result set #2 encontrado: 24 linhas

[✓] Query retornou 24 linhas com 25 colunas

CONSULTA EXECUTADA COM SUCESSO!
Total de linhas retornadas: 24
Colunas: ['DATA', 'FAIXA', 'Assigned_Portfolio', 'Reachable_Portfolio', 'Contacted_Portfolio', 'Active_Agents', 'Total_Call_Attempts', 'Total_Answered_Calls', 'Dont_Know_on_the_Phone', 'Total_Answered_Calls_Excl_Short_Calls', 'U

In [ ]:
# Verifica quantas datas únicas existem
datas_unicas = df_resultado['DATA'].unique()
print(f"Datas encontradas: {datas_unicas}")
print(f"Total de datas: {len(datas_unicas)}")

# Define automaticamente as colunas de indicadores
# Todas as colunas EXCETO 'DATA' e 'FAIXA'
colunas_todas = df_resultado.columns.tolist()
colunas_id = ['DATA', 'FAIXA']  # Colunas identificadoras
value_columns = [col for col in colunas_todas if col not in colunas_id]

print(f"\nColunas identificadoras: {colunas_id}")
print(f"Total de indicadores encontrados: {len(value_columns)}")

# Converte a coluna DATA para o formato DD/MM/YYYY
df_resultado['DATA'] = pd.to_datetime(df_resultado['DATA']).dt.strftime('%d/%m/%Y')

# Faz o unpivot (melt) mantendo DATA e FAIXA como identificadores
df_transposto = df_resultado.melt(
    id_vars=['DATA', 'FAIXA'],
    value_vars=value_columns,
    var_name='Indicador',
    value_name='Valor'
)

print(f"\nDataFrame após melt:")
print(df_transposto.head(10))

# Pivota para colocar as datas como colunas
df_final = df_transposto.pivot_table(
    index=['FAIXA', 'Indicador'],
    columns='DATA',
    values='Valor',
    aggfunc='first'  # Usa o primeiro valor caso haja duplicatas
).reset_index()

# Remove o nome do índice das colunas (fica mais limpo)
df_final.columns.name = None

# Ordena as colunas de data cronologicamente
colunas_fixas = ['FAIXA', 'Indicador']
colunas_data = [col for col in df_final.columns if col not in colunas_fixas]

# Ordena as datas convertendo de volta para datetime temporariamente
colunas_data_ordenadas = sorted(colunas_data, key=lambda x: pd.to_datetime(x, format='%d/%m/%Y'))

# Reorganiza o DataFrame com as colunas ordenadas
df_final = df_final[colunas_fixas + colunas_data_ordenadas]

# Ordena por FAIXA e Indicador
df_final = df_final.sort_values(['FAIXA', 'Indicador']).reset_index(drop=True)

# Visualiza o resultado
print(f"\n{'='*60}")
print(f"Estrutura final:")
print(f"Total de linhas: {len(df_final)}")
print(f"Faixas únicas: {df_final['FAIXA'].unique()}")
print(f"Colunas: {df_final.columns.tolist()}")
print(f"{'='*60}")
print("\nPrimeiras 30 linhas:")
print(df_final.head(30))

df_final

Datas encontradas: [datetime.date(2026, 1, 8) datetime.date(2026, 1, 9)
 datetime.date(2026, 1, 12) datetime.date(2026, 1, 13)
 datetime.date(2026, 1, 14) datetime.date(2026, 1, 15)]
Total de datas: 6

Colunas identificadoras: ['DATA', 'FAIXA']
Total de indicadores encontrados: 23

DataFrame após melt:
         DATA       FAIXA           Indicador  Valor
0  08/01/2026   DPD 61-90  Assigned_Portfolio   2094
1  08/01/2026  DPD 91-120  Assigned_Portfolio   1112
2  08/01/2026   DPD 16-30  Assigned_Portfolio   2782
3  08/01/2026   DPD 31-60  Assigned_Portfolio   2224
4  09/01/2026  DPD 91-120  Assigned_Portfolio   1112
5  09/01/2026   DPD 16-30  Assigned_Portfolio   2782
6  09/01/2026   DPD 31-60  Assigned_Portfolio   2224
7  09/01/2026   DPD 61-90  Assigned_Portfolio   2094
8  12/01/2026   DPD 16-30  Assigned_Portfolio   2782
9  12/01/2026   DPD 61-90  Assigned_Portfolio   2094

Estrutura final:
Total de linhas: 92
Faixas únicas: ['DPD 16-30' 'DPD 31-60' 'DPD 61-90' 'DPD 91-120']
Colunas: 

,FAIXA,Indicador,08/01/2026,09/01/2026,12/01/2026,13/01/2026,14/01/2026,15/01/2026
0,DPD 16-30,AHT_Excluding_Short_Calls,2248,3569,1995,1351,2057,2779
1,DPD 16-30,Active_Agents,4,4,4,4,4,4
2,DPD 16-30,Assigned_Portfolio,2782,2782,2782,2782,2782,2782
3,DPD 16-30,Contacted_Portfolio,1676,1587,1698,1544,2211,2416
4,DPD 16-30,Debt_Not_Recognized,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...
87,DPD 91-120,Total_Call_Attempts,1433,1376,1334,1244,948,1449
88,DPD 91-120,Total_RPCs_Right_Party_Contacts,6,2,4,3,1,0
89,DPD 91-120,Unique_Customers_Reached,31,30,30,22,23,30
90,DPD 91-120,Unique_Customers_Reached_Excl_Short_Calls,17,14,18,13,13,17


In [18]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append("..") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import pandas as pd
from utils.db_connection import get_connection

def df_contratos_payjoy():
    """
    Busca os contratos PayJoy com suas respectivas faixas (DPD_BUCKET)
    
    Returns:
        DataFrame com colunas: Assigned_Portfolio, FAIXA
    """
    # Cria a conexão
    conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
    
    # Query SQL
    query = """
    SELECT
        Assigned_Portfolio
    ,   DPD_BUCKET as FAIXA
    FROM OPENQUERY([TRC_BD_LINKED],
    '
        SELECT 
            ltrim(rtrim(contrato_fin)) Assigned_Portfolio
        ,   DPD_BUCKET
        ,   cast(CALENDAR_DATE as date) dt_base 
        FROM SRC..AUX_PAYJOY_REMESSA 
        WHERE CALENDAR_DATE = (SELECT MAX(CALENDAR_DATE) FROM SRC..AUX_PAYJOY_REMESSA)
    '
    )
    """
    
    try:
        # Executa a query e retorna o DataFrame
        df = pd.read_sql(query, conn_bd2)
        
        print(f"Total de contratos: {len(df)}")
        print(f"Faixas disponíveis: {df['FAIXA'].unique()}")
        
        return df
        
    except Exception as e:
        print(f"Erro ao executar query: {e}")
        raise
    finally:
        conn_bd2.close()

# Executa a função
df_contratos = df_contratos_payjoy()
df_contratos

Total de contratos: 8212
Faixas disponíveis: ['DPD 31-60' 'DPD 16-30' 'DPD 61-90' 'DPD 91-120']


C:\Users\claudiano.alves\AppData\Local\Temp\ipykernel_23228\2811541349.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn_bd2)


,Assigned_Portfolio,FAIXA
0,55577709ID,DPD 31-60
1,55577710ID,DPD 16-30
2,55577711ID,DPD 31-60
3,55577712ID,DPD 31-60
4,55577713ID,DPD 31-60
...,...,...
8207,55665838ID,DPD 16-30
8208,55665839ID,DPD 16-30
8209,55665840ID,DPD 16-30
8210,55665841ID,DPD 16-30


In [25]:
import pandas as pd
import os
from glob import glob
import re

def consolidar_mailings_payjoy(caminho_pasta=r"\\trc-dc-ad\Planejamento\00 - USUÁRIOS\0003_Daniel Kodama\PAYJOY\mailings payjoy"):
    """
    Consolida todos os arquivos CSV de mailings PayJoy em um único DataFrame
    """
    
    arquivos = glob(os.path.join(caminho_pasta, "*.csv"))
    
    if not arquivos:
        print(f"Nenhum arquivo CSV encontrado em: {caminho_pasta}")
        return pd.DataFrame()
    
    print(f"Encontrados {len(arquivos)} arquivos CSV\n")
    
    lista_dfs = []
    arquivos_com_erro = []
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            nome_sem_extensao = nome_arquivo.replace('.csv', '')
            
            # Busca padrão de data no nome: DD.MM ou DD/MM
            match = re.search(r'(\d{1,2})[./](\d{2})', nome_sem_extensao)
            
            if match:
                dia = match.group(1).zfill(2)  # Adiciona zero à esquerda se necessário
                mes = match.group(2)
                data_str = f"{dia}/{mes}"
            else:
                # Se não encontrar data, usa os últimos 4 caracteres (método antigo)
                ultimos_4_chars = nome_sem_extensao[-4:]
                data_str = ultimos_4_chars.replace('.', '/')
            
            # Tenta diferentes combinações de encoding e separador
            df_temp = None
            for sep in [';', ',', '\t']:
                for enc in ['latin-1', 'utf-8', 'cp1252']:
                    try:
                        df_temp = pd.read_csv(arquivo, usecols=['CONTRATO'], encoding=enc, sep=sep)
                        if not df_temp.empty and 'CONTRATO' in df_temp.columns:
                            break
                    except:
                        continue
                if df_temp is not None and not df_temp.empty:
                    break
            
            if df_temp is None or df_temp.empty:
                raise Exception("Não foi possível ler o arquivo com nenhuma combinação")
            
            df_temp['DATA'] = data_str
            lista_dfs.append(df_temp)
            
            print(f"✓ {nome_arquivo} - {len(df_temp)} registros - Data: {data_str}")
            
        except Exception as e:
            print(f"✗ ERRO em {nome_arquivo}: {e}")
            arquivos_com_erro.append((nome_arquivo, str(e)))
            continue
    
    if arquivos_com_erro:
        print(f"\n{'='*60}")
        print("ARQUIVOS COM ERRO:")
        for nome, erro in arquivos_com_erro:
            print(f"  - {nome}: {erro}")
        print(f"{'='*60}\n")
    
    if lista_dfs:
        df_consolidado = pd.concat(lista_dfs, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"Total de registros consolidados: {len(df_consolidado)}")
        print(f"Datas únicas: {sorted(df_consolidado['DATA'].unique())}")
        print(f"Total de contratos únicos: {df_consolidado['CONTRATO'].nunique()}")
        print(f"{'='*60}")
        
        return df_consolidado
    else:
        print("Nenhum dado foi consolidado.")
        return pd.DataFrame()

import logging
import pandas as pd
import os
from glob import glob
import re

log_level = logging.DEBUG if os.getenv('PAYJOY_DEBUG', '0') == '1' else logging.INFO
logging.basicConfig(level=log_level, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

def consolidar_mailings_payjoy_(caminho_pasta=r"\\trc-dc-ad\Planejamento\00 - USUÁRIOS\0003_Daniel Kodama\PAYJOY\mailings payjoy"):
    """
    Consolida todos os arquivos CSV de mailings PayJoy em um único DataFrame
    """
    
    arquivos = glob(os.path.join(caminho_pasta, "*.csv"))
    
    if not arquivos:
        logger.info(f"Nenhum arquivo CSV encontrado em: {caminho_pasta}")
        return pd.DataFrame()
    
    logger.info(f"Encontrados {len(arquivos)} arquivos CSV")
    
    lista_dfs = []
    arquivos_com_erro = []
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            nome_sem_extensao = nome_arquivo.replace('.csv', '')
            
            # Busca padrão de data no nome: DD.MM ou DD/MM ou DD_MM
            # Trabalha com separadores: . / _
            match = re.search(r'(\d{1,2})[./\_](\d{1,2})', nome_sem_extensao)
            
            if match:
                dia = match.group(1).zfill(2)  # Adiciona zero à esquerda se necessário
                mes = match.group(2).zfill(2)  # Adiciona zero à esquerda se necessário
                data_str = f"{dia}/{mes}"
                logger.debug(f"  Data extraída: {match.group(0)} -> {data_str}")
            else:
                # Se não encontrar data, usa os últimos 4 caracteres (método antigo)
                ultimos_4_chars = nome_sem_extensao[-4:]
                data_str = ultimos_4_chars.replace('.', '/').replace('_', '/')
                logger.warning(f"  ⚠ Padrão DD/MM não encontrado em '{nome_sem_extensao}', usando fallback: {data_str}")
            
            # Tenta identificar se o arquivo traz 'CHAVE_POPUP' ou 'CONTRATO' e lê apenas essa coluna
            df_temp = None
            col_found = None
            sep_used = None
            enc_used = None

            def _normalize_col(c):
                return re.sub(r'[^A-Z0-9]', '', str(c).upper())

            for sep in [';', ',', '\t']:
                for enc in ['latin-1', 'utf-8', 'cp1252']:
                    try:
                        # lê apenas header para detectar colunas
                        df_try = pd.read_csv(arquivo, nrows=0, encoding=enc, sep=sep)
                        cols = [_normalize_col(c) for c in df_try.columns.tolist()]

                        if 'CHAVEPOPUP' in cols:
                            col_found = df_try.columns[cols.index('CHAVEPOPUP')]
                        elif 'CONTRATO' in cols:
                            col_found = df_try.columns[cols.index('CONTRATO')]

                        if col_found is not None:
                            sep_used = sep
                            enc_used = enc
                            break
                    except Exception:
                        continue
                if col_found is not None:
                    break

            if col_found is None:
                raise Exception("Não foi possível ler o arquivo com as colunas esperadas (CHAVE_POPUP ou CONTRATO)")

            # Lê apenas a coluna identificada
            try:
                df_temp = pd.read_csv(arquivo, usecols=[col_found], encoding=enc_used, sep=sep_used)
            except Exception as e:
                raise Exception(f"Falha ao ler coluna '{col_found}' do arquivo: {e}")

            # Normaliza para coluna padrão 'CONTRATO'
            if col_found != 'CONTRATO':
                df_temp = df_temp.rename(columns={col_found: 'CONTRATO'})

            df_temp = df_temp[['CONTRATO']].copy()
            df_temp['DATA'] = data_str
            lista_dfs.append(df_temp)

            logger.debug(f"✓ {nome_arquivo} - {len(df_temp)} registros - Data: {data_str} - Col: {col_found}")
            
        except Exception as e:
            print(f"✗ ERRO em {nome_arquivo}: {e}")
            arquivos_com_erro.append((nome_arquivo, str(e)))
            continue
    
    if arquivos_com_erro:
        logger.warning('\n' + '='*60)
        logger.warning('ARQUIVOS COM ERRO:')
        for nome, erro in arquivos_com_erro:
            logger.warning(f"  - {nome}: {erro}")
        logger.warning('='*60)
    
    if lista_dfs:
        df_consolidado = pd.concat(lista_dfs, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"Total de registros consolidados: {len(df_consolidado)}")
        print(f"Datas únicas: {sorted(df_consolidado['DATA'].unique())}")
        if 'CONTRATO' in df_consolidado.columns:
            unique_count = df_consolidado['CONTRATO'].nunique()
        else:
            unique_count = df_consolidado.iloc[:,0].nunique()

        print(f"Total de contratos únicos: {unique_count}")
        print(f"{'='*60}")

        return df_consolidado
    else:
        print("Nenhum dado foi consolidado.")
        return pd.DataFrame()

df_mailings_ = consolidar_mailings_payjoy_()

df_mailings = consolidar_mailings_payjoy()
#df_mailings
df_mailings_

[INFO] Encontrados 11 arquivos CSV



Total de registros consolidados: 90800
Datas únicas: ['02/01', '05/01', '06/01', '07/01', '08/01', '09/01', '12/01', '13/01', '14/01', '15/01', '16/01']
Total de contratos únicos: 17395
Encontrados 11 arquivos CSV

✓ PAYJOY-ATIVO_02.01.csv - 8399 registros - Data: 02/01
✓ PAYJOY-ATIVO_05.01.csv - 8399 registros - Data: 05/01
✓ PAYJOY-ATIVO_06_01.csv - 8399 registros - Data: 6_01
✓ PAYJOY-ATIVO_07_01.csv - 8399 registros - Data: 7_01
✓ PAYJOY-ATIVO_08_01.csv - 8141 registros - Data: 8_01
✓ PAYJOY-ATIVO_09_01.csv - 8141 registros - Data: 9_01
✓ PAYJOY-ATIVO_12.01.csv - 8141 registros - Data: 12/01
✓ PAYJOY-ATIVO_13_01.csv - 8142 registros - Data: 3_01
✓ PAYJOY-ATIVO_14_01.csv - 8213 registros - Data: 4_01
✓ PAYJOY-ATIVO_15_01.csv - 8213 registros - Data: 5_01
✓ PAYJOY-ATIVO_16_01.csv - 8213 registros - Data: 6_01

Total de registros consolidados: 90800
Datas únicas: ['02/01', '05/01', '12/01', '3_01', '4_01', '5_01', '6_01', '7_01', '8_01', '9_01']
Total de contratos únicos: 17395


,CONTRATO,DATA
0,55154266ID,02/01
1,55153637ID,02/01
2,55154323ID,02/01
3,55154458ID,02/01
4,55154329ID,02/01
...,...,...
90795,55583037ID,16/01
90796,55580048ID,16/01
90797,55582707ID,16/01
90798,55578880ID,16/01


In [50]:
def consolidar_mailings_por_faixa__(df_mailings, df_contratos, ordenar_datas=True, contratos_unicos=True):
    """
    Cruza os mailings com os contratos para obter faixas e gera tabela pivotada
    """
    
    # LIMPEZA: Remove espaços em branco das colunas de chave
    df_mailings['CONTRATO'] = df_mailings['CONTRATO'].astype(str).str.strip()
    df_contratos['Assigned_Portfolio'] = df_contratos['Assigned_Portfolio'].astype(str).str.strip()
    
    # Cruzamento
    df_mailings_com_faixa = df_mailings.merge(
        df_contratos,
        left_on='CONTRATO',
        right_on='Assigned_Portfolio',
        how='left'
    )
    
    # Contagem
    if contratos_unicos:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])['CONTRATO'].nunique().reset_index()
    else:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])['CONTRATO'].count().reset_index()
    
    df_contagem.columns = ['FAIXA', 'DATA', 'CONTAGEM']
    
    # Pivota
    df_pivot = df_contagem.pivot(
        index='FAIXA',
        columns='DATA',
        values='CONTAGEM'
    ).fillna(0).astype(int)
    
    # Ordena datas se solicitado
    if ordenar_datas:
        def ordenar_data(data_str):
            """Converte DD/MM para tupla (dia, mes) para ordenação"""
            try:
                dia, mes = data_str.split('/')
                return (int(mes), int(dia))
            except:
                return (99, 99)
        
        colunas_ordenadas = sorted(df_pivot.columns, key=ordenar_data)
        df_pivot = df_pivot[colunas_ordenadas]
    
    # Reset index
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

def consolidar_mailings_por_faixa(df_mailings, df_contratos, ordenar_datas=True, contratos_unicos=True):
    """
    Cruza os mailings com os contratos para obter faixas e gera tabela pivotada
    """
    
    # NORMALIZAÇÃO: aceita tanto 'CONTRATO' quanto 'CHAVE_POPUP' como coluna de chave
    if 'CONTRATO' in df_mailings.columns:
        mail_col = 'CONTRATO'
    elif 'CHAVE_POPUP' in df_mailings.columns:
        mail_col = 'CHAVE_POPUP'
    else:
        # Pega a primeira coluna disponível como último recurso
        mail_col = df_mailings.columns[0]
        logger.warning(f"Nenhuma coluna 'CONTRATO' ou 'CHAVE_POPUP' encontrada em df_mailings. Usando '{mail_col}' como chave.")

    # LIMPEZA: Remove espaços em branco das colunas de chave
    df_mailings[mail_col] = df_mailings[mail_col].astype(str).str.strip()
    df_contratos['Assigned_Portfolio'] = df_contratos['Assigned_Portfolio'].astype(str).str.strip()

    # Cruzamento usando a coluna detectada
    df_mailings_com_faixa = df_mailings.merge(
        df_contratos,
        left_on=mail_col,
        right_on='Assigned_Portfolio',
        how='left'
    )

    # Substitui FAIXA vazia por 'SEM_FAIXA' para evitar perda de linhas durante agrupamento
    df_mailings_com_faixa['FAIXA'] = df_mailings_com_faixa['FAIXA'].fillna('SEM_FAIXA')

    # AJUSTE: Adiciona ano 2026 à coluna DATA se ainda não tiver
    def adicionar_ano(data_str):
        """Adiciona /2026 se a data estiver no formato DD/MM"""
        data_str = str(data_str).strip()
        if '/' in data_str:
            partes = data_str.split('/')
            if len(partes) == 2:  # formato DD/MM
                return f"{partes[0]}/{partes[1]}/2026"
            elif len(partes) == 3:  # formato DD/MM/YYYY
                return data_str
        return data_str
    
    df_mailings_com_faixa['DATA'] = df_mailings_com_faixa['DATA'].apply(adicionar_ano)

    # Contagem
    if contratos_unicos:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])[mail_col].nunique().reset_index()
    else:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])[mail_col].count().reset_index()

    df_contagem.columns = ['FAIXA', 'DATA', 'CONTAGEM']
    
    # Pivota
    df_pivot = df_contagem.pivot(
        index='FAIXA',
        columns='DATA',
        values='CONTAGEM'
    ).fillna(0).astype(int)
    
    # Ordena datas se solicitado
    if ordenar_datas:
        def ordenar_data(data_str):
            """Converte DD/MM/YYYY para tupla (ano, mes, dia) para ordenação"""
            try:
                partes = data_str.split('/')
                if len(partes) == 3:  # DD/MM/YYYY
                    return (int(partes[2]), int(partes[1]), int(partes[0]))
                elif len(partes) == 2:  # DD/MM (fallback)
                    return (2026, int(partes[1]), int(partes[0]))
                else:
                    return (9999, 99, 99)
            except:
                return (9999, 99, 99)
        
        colunas_ordenadas = sorted(df_pivot.columns, key=ordenar_data)
        df_pivot = df_pivot[colunas_ordenadas]
    
    # Reset index
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

# Uso:
df_pivot = consolidar_mailings_por_faixa(df_mailings_, df_contratos)
df_pivot

DATA,FAIXA,02/01/2026,05/01/2026,06/01/2026,07/01/2026,08/01/2026,09/01/2026,12/01/2026,13/01/2026,14/01/2026,15/01/2026,16/01/2026
0,DPD 16-30,0,0,0,0,1928,1928,1928,1928,2782,2782,2782
1,DPD 31-60,0,0,0,0,2224,2224,2224,2224,2224,2224,2224
2,DPD 61-90,0,0,0,0,2094,2094,2094,2094,2094,2094,2094
3,DPD 91-120,0,0,0,0,1112,1112,1112,1112,1112,1112,1112
4,SEM_FAIXA,8399,8399,8399,8399,783,783,783,784,1,1,1


In [51]:
import pandas as pd

def substituir_zeros_com_pivot_(df_transposto, df_pivot):
    """
    Substitui valores 0 do df_transposto pelos valores do df_pivot,
    respeitando FAIXA + data.
    
    Args:
        df_transposto: DataFrame com estrutura [FAIXA, Indicador, 2025-12-04, 2025-12-05, ...]
        df_pivot: DataFrame com estrutura [FAIXA, 3/12, 4/12, 5/12, ...]
    
    Returns:
        DataFrame atualizado
    """
    
    df_resultado = df_transposto.copy()
    
    # Pegar as colunas de data do df_transposto
    colunas_data = [col for col in df_transposto.columns if col not in ['FAIXA', 'Indicador']]
    
    # Para cada linha do df_pivot
    for _, row_pivot in df_pivot.iterrows():
        faixa = row_pivot["FAIXA"]
        
        # Para cada coluna de data no df_transposto
        for col_transposto in colunas_data:
            # Converter coluna do df_transposto para formato do df_pivot
            if isinstance(col_transposto, str) and '-' in col_transposto:
                partes = col_transposto.split('-')
                dia = partes[2].lstrip('0') or '0'
                mes = partes[1].lstrip('0') or '0'
                col_pivot_format = f"{dia}/{mes}"
            elif hasattr(col_transposto, 'day') and hasattr(col_transposto, 'month'):
                dia = str(col_transposto.day)
                mes = str(col_transposto.month)
                col_pivot_format = f"{dia}/{mes}"
            else:
                continue
            
            # Se essa coluna existe no df_pivot
            if col_pivot_format in df_pivot.columns:
                valor_pivot = row_pivot[col_pivot_format]
                
                # Substituir os zeros APENAS para o indicador Reachable_Portfolio
                mask = (
                    (df_resultado["FAIXA"] == faixa) & 
                    (df_resultado["Indicador"] == "Reachable_Portfolio") &
                    (df_resultado[col_transposto] == 0)
                )
                df_resultado.loc[mask, col_transposto] = valor_pivot
    
    return df_resultado


def substituir_zeros_com_pivot_(df_final, df_pivot, debug=True):
    """
    Substitui valores 0 do df_final pelos valores do df_pivot,
    respeitando FAIXA + data.
    
    Args:
        df_final: DataFrame com estrutura [FAIXA, Indicador, 2025-12-04, 2025-12-05, ...]
        df_pivot: DataFrame com estrutura [FAIXA, 3/12, 4/12, 5/12, ...]
        debug: Se True, imprime informações de debug
    
    Returns:
        DataFrame atualizado
    """

    df_resultado = df_final.copy()

    if debug:
        print("="*80)
        print("DEBUG: Estrutura dos DataFrames")
        print("="*80)
        print(f"\ndf_final - Colunas: {df_final.columns.tolist()}")
        print(f"df_final - Shape: {df_final.shape}")
        print(f"\ndf_pivot - Colunas: {df_pivot.columns.tolist()}")
        print(f"df_pivot - Shape: {df_pivot.shape}")
        print(f"\nFaixas em df_final: {df_final['FAIXA'].unique()}")
        print(f"Faixas em df_pivot: {df_pivot['FAIXA'].unique()}")
        print(f"\nIndicadores em df_final: {df_final['Indicador'].unique()}")
        print("\nPrimeiras linhas df_final:")
        print(df_final.head())
        print("\nPrimeiras linhas df_pivot:")
        print(df_pivot.head())
    
    # Pegar as colunas de data do df_final
    colunas_data = [col for col in df_final.columns if col not in ['FAIXA', 'Indicador']]
    
    if debug:
        print(f"\n{'='*80}")
        print(f"Colunas de data identificadas: {colunas_data[:5]}... (total: {len(colunas_data)})")
        print(f"{'='*80}\n")
    
    substituicoes_realizadas = 0
    
    # Para cada linha do df_pivot
    for idx_pivot, row_pivot in df_pivot.iterrows():
        faixa = row_pivot["FAIXA"]
        
        if debug and idx_pivot == 0:
            print(f"\nProcessando FAIXA: {faixa}")
        
        # Para cada coluna de data no df_final
        for col_final in colunas_data:
            # Converter coluna do df_final para formato do df_pivot
            if isinstance(col_final, str):
                # Tenta diferentes formatos de data
                if '-' in col_final:  # Formato: 2025-12-04
                    partes = col_final.split('-')
                    dia = partes[2].lstrip('0') or '0'
                    mes = partes[1].lstrip('0') or '0'
                    ano = partes[0]
                    col_pivot_format = f"{dia}/{mes}/2026"  # Ajustado para incluir ano
                elif '/' in col_final:  # Formato: 04/12/2025
                    partes = col_final.split('/')
                    dia = partes[0].lstrip('0') or '0'
                    mes = partes[1].lstrip('0') or '0'
                    col_pivot_format = f"{dia}/{mes}/2026"
                else:
                    continue
            elif hasattr(col_final, 'day') and hasattr(col_final, 'month'):
                dia = str(col_final.day)
                mes = str(col_final.month)
                col_pivot_format = f"{dia}/{mes}/2026"
            else:
                continue
            
            # Se essa coluna existe no df_pivot
            if col_pivot_format in df_pivot.columns:
                valor_pivot = row_pivot[col_pivot_format]
                
                # Substituir os zeros APENAS para o indicador Reachable_Portfolio
                mask = (
                    (df_resultado["FAIXA"] == faixa) & 
                    (df_resultado["Indicador"] == "Reachable_Portfolio") &
                    (df_resultado[col_final] == 0)
                )
                
                linhas_afetadas = mask.sum()
                
                if linhas_afetadas > 0:
                    df_resultado.loc[mask, col_final] = valor_pivot
                    substituicoes_realizadas += linhas_afetadas
                    
                    if debug and idx_pivot == 0 and substituicoes_realizadas <= 3:
                        print(f"  ✓ Substituído: {col_final} -> {col_pivot_format}")
                        print(f"    Valor: 0 -> {valor_pivot}")
                        print(f"    Linhas afetadas: {linhas_afetadas}")
            elif debug and idx_pivot == 0 and col_final == colunas_data[0]:
                print(f"  ✗ Coluna não encontrada no pivot: {col_pivot_format}")
                print(f"    Colunas disponíveis no pivot: {[c for c in df_pivot.columns if '/' in str(c)][:5]}")
    
    if debug:
        print(f"\n{'='*80}")
        print(f"Total de substituições realizadas: {substituicoes_realizadas}")
        print(f"{'='*80}\n")
    
    return df_resultado

def substituir_zeros_com_pivot(df_transposto, df_pivot):
    """
    Substitui valores 0 do df_transposto pelos valores do df_pivot,
    respeitando FAIXA + data.
    
    Args:
        df_transposto: DataFrame com estrutura [DATA, FAIXA, Indicador, Valor] (formato longo)
        df_pivot: DataFrame com estrutura [FAIXA, 02/01/2026, 05/01/2026, ...] (formato largo)
    
    Returns:
        DataFrame atualizado no formato original
    """
    
    print("Iniciando substituição...")
    
    # Verificar estrutura
    if 'DATA' in df_transposto.columns and 'Valor' in df_transposto.columns:
        # DataFrame está em formato longo - converter para largo primeiro
        df_largo = df_transposto.pivot_table(
            index=['FAIXA', 'Indicador'],
            columns='DATA',
            values='Valor',
            aggfunc='first'
        ).reset_index()
        
        # Remove o nome do índice das colunas
        df_largo.columns.name = None
        
        print(f"Convertido para formato largo: {df_largo.shape}")
        print(f"Colunas: {df_largo.columns.tolist()[:5]}...")
        
        formato_longo = True
        df_trabalho = df_largo.copy()
    else:
        # Já está em formato largo
        formato_longo = False
        df_trabalho = df_transposto.copy()
    
    # Pegar as colunas de data
    colunas_data = [col for col in df_trabalho.columns if col not in ['FAIXA', 'Indicador']]
    
    print(f"Colunas de data identificadas: {len(colunas_data)}")
    print(f"Primeiras datas: {colunas_data[:3]}")
    
    substituicoes = 0
    
    # Para cada linha do df_pivot
    for _, row_pivot in df_pivot.iterrows():
        faixa = row_pivot["FAIXA"]
        
        # Para cada coluna de data no df_pivot (exceto FAIXA)
        for col_pivot in df_pivot.columns:
            if col_pivot == 'FAIXA':
                continue
            
            # Verificar se essa data existe no df_trabalho
            if col_pivot in colunas_data:
                valor_pivot = row_pivot[col_pivot]
                
                # Substituir os zeros APENAS para o indicador Reachable_Portfolio
                mask = (
                    (df_trabalho["FAIXA"] == faixa) & 
                    (df_trabalho["Indicador"] == "Reachable_Portfolio") &
                    (df_trabalho[col_pivot] == 0)
                )
                
                linhas_afetadas = mask.sum()
                
                if linhas_afetadas > 0:
                    df_trabalho.loc[mask, col_pivot] = valor_pivot
                    substituicoes += 1
                    
                    if substituicoes <= 3:  # Mostra apenas as primeiras 3
                        print(f"  ✓ FAIXA {faixa}, Data {col_pivot}: 0 -> {valor_pivot}")
    
    print(f"\nTotal de substituições: {substituicoes}")
    
    # Se estava em formato longo, converter de volta
    if formato_longo:
        # Converter de volta para formato longo
        df_resultado = df_trabalho.melt(
            id_vars=['FAIXA', 'Indicador'],
            var_name='DATA',
            value_name='Valor'
        )
        
        # Ordenar como estava antes
        df_resultado = df_resultado.sort_values(['DATA', 'FAIXA', 'Indicador']).reset_index(drop=True)
        
        print(f"Convertido de volta para formato longo: {df_resultado.shape}")
        
        return df_resultado
    else:
        return df_trabalho

# Usar assim:
df_resultado_atualizado = substituir_zeros_com_pivot(df_transposto, df_pivot)

def substituir_zeros_com_pivot_debug(df_final, df_pivot):
    """
    Versão debug completa para identificar o problema
    """
    
    df_resultado = df_final.copy()
    
    print("="*80)
    print("ANÁLISE DETALHADA")
    print("="*80)
    
    # 1. Verificar estrutura básica
    print("\n1. ESTRUTURA DOS DATAFRAMES:")
    print(f"   df_transposto shape: {df_transposto.shape}")
    print(f"   df_pivot shape: {df_pivot.shape}")
    
    # 2. Verificar colunas
    print("\n2. COLUNAS:")
    print(f"   df_transposto: {df_transposto.columns.tolist()}")
    print(f"   df_pivot: {df_pivot.columns.tolist()}")
    
    # 3. Verificar se existe o indicador Reachable_Portfolio
    if 'Indicador' in df_transposto.columns:
        indicadores = df_transposto['Indicador'].unique()
        print(f"\n3. INDICADORES em df_transposto: {indicadores}")
        tem_reachable = 'Reachable_Portfolio' in indicadores
        print(f"   Tem 'Reachable_Portfolio'? {tem_reachable}")
        
        if tem_reachable:
            # Mostrar dados do Reachable_Portfolio
            df_reach = df_transposto[df_transposto['Indicador'] == 'Reachable_Portfolio']
            print(f"\n   Linhas com Reachable_Portfolio: {len(df_reach)}")
            print("\n   Primeiras linhas:")
            print(df_reach.head())
    else:
        print("\n3. ERRO: Coluna 'Indicador' não encontrada!")
        return df_resultado
    
    # 4. Verificar faixas
    print("\n4. FAIXAS:")
    faixas_transp = set(df_transposto['FAIXA'].unique())
    faixas_pivot = set(df_pivot['FAIXA'].unique())
    print(f"   df_transposto: {faixas_transp}")
    print(f"   df_pivot: {faixas_pivot}")
    print(f"   Faixas em comum: {faixas_transp.intersection(faixas_pivot)}")
    
    # 5. Pegar colunas de data
    colunas_data = [col for col in df_transposto.columns if col not in ['FAIXA', 'Indicador']]
    print(f"\n5. COLUNAS DE DATA em df_transposto: {len(colunas_data)}")
    print(f"   Primeiras 3: {colunas_data[:3]}")
    print(f"   Tipos: {[type(col).__name__ for col in colunas_data[:3]]}")
    
    # 6. Testar conversão de data
    print("\n6. TESTE DE CONVERSÃO DE DATA:")
    for i, col in enumerate(colunas_data[:3]):
        print(f"\n   Coluna {i+1}: {col} (tipo: {type(col).__name__})")
        
        # Tentar diferentes conversões
        if isinstance(col, str):
            if '-' in col:
                partes = col.split('-')
                print(f"      Partes (split por '-'): {partes}")
                dia = partes[2].lstrip('0') or '0'
                mes = partes[1].lstrip('0') or '0'
                conversao = f"{dia}/{mes}/2026"
                print(f"      Conversão: {conversao}")
                print(f"      Existe no pivot? {conversao in df_pivot.columns}")
            elif '/' in col:
                partes = col.split('/')
                print(f"      Partes (split por '/'): {partes}")
                dia = partes[0].lstrip('0') or '0'
                mes = partes[1].lstrip('0') or '0'
                conversao = f"{dia}/{mes}/2026"
                print(f"      Conversão: {conversao}")
                print(f"      Existe no pivot? {conversao in df_pivot.columns}")
        elif hasattr(col, 'strftime'):
            conversao = col.strftime('%d/%m/%Y')
            print(f"      Conversão (datetime): {conversao}")
            print(f"      Existe no pivot? {conversao in df_pivot.columns}")
    
    # 7. Verificar valores zeros
    print("\n7. VERIFICAÇÃO DE ZEROS em Reachable_Portfolio:")
    if tem_reachable:
        for col in colunas_data[:3]:
            mask_reach = df_transposto['Indicador'] == 'Reachable_Portfolio'
            valores = df_transposto.loc[mask_reach, col]
            zeros = (valores == 0).sum()
            print(f"   Coluna {col}: {zeros} zeros de {len(valores)} valores")
            print(f"      Valores únicos: {valores.unique()[:5]}")
    
    # 8. Teste real de substituição em uma faixa
    print("\n8. TESTE DE SUBSTITUIÇÃO (primeira faixa):")
    if len(df_pivot) > 0:
        faixa_teste = df_pivot.iloc[0]['FAIXA']
        print(f"   Testando com FAIXA: {faixa_teste}")
        
        col_teste = colunas_data[0]
        print(f"   Testando com coluna: {col_teste}")
        
        # Converter formato
        if isinstance(col_teste, str) and '/' in col_teste:
            partes = col_teste.split('/')
            col_pivot_format = f"{partes[0].lstrip('0') or '0'}/{partes[1].lstrip('0') or '0'}/2026"
        else:
            col_pivot_format = "CONVERSÃO FALHOU"
        
        print(f"   Formato convertido: {col_pivot_format}")
        print(f"   Existe no pivot? {col_pivot_format in df_pivot.columns}")
        
        if col_pivot_format in df_pivot.columns:
            valor_pivot = df_pivot[df_pivot['FAIXA'] == faixa_teste][col_pivot_format].values
            print(f"   Valor no pivot: {valor_pivot}")
            
            # Testar máscara
            mask = (
                (df_resultado["FAIXA"] == faixa_teste) & 
                (df_resultado["Indicador"] == "Reachable_Portfolio") &
                (df_resultado[col_teste] == 0)
            )
            print(f"   Linhas que atendem a máscara: {mask.sum()}")
            
            # Verificar cada condição separadamente
            mask1 = df_resultado["FAIXA"] == faixa_teste
            mask2 = df_resultado["Indicador"] == "Reachable_Portfolio"
            mask3 = df_resultado[col_teste] == 0
            
            print(f"\n   Análise das condições:")
            print(f"      FAIXA == {faixa_teste}: {mask1.sum()} linhas")
            print(f"      Indicador == 'Reachable_Portfolio': {mask2.sum()} linhas")
            print(f"      Valor == 0: {mask3.sum()} linhas")
            print(f"      Todas as 3 condições: {mask.sum()} linhas")
    
    print("\n" + "="*80)
    
    return df_resultado

# Execute esta versão
# df_debug = substituir_zeros_com_pivot_debug(df_final, df_pivot)
# df_debug
# Exemplo de uso
# df_novo = substituir_zeros_com_pivot(df_transposto, df_pivot, debug=True)
df_PAYJOY = substituir_zeros_com_pivot(df_final, df_pivot)
print("DataFrame atualizado:")
df_filtrado = df_PAYJOY[df_PAYJOY["Indicador"] == "Reachable_Portfolio"]
df_filtrado

Iniciando substituição...
Convertido para formato largo: (92, 8)
Colunas: ['FAIXA', 'Indicador', '08/01/2026', '09/01/2026', '12/01/2026']...
Colunas de data identificadas: 6
Primeiras datas: ['08/01/2026', '09/01/2026', '12/01/2026']
  ✓ FAIXA DPD 16-30, Data 08/01/2026: 0 -> 1928
  ✓ FAIXA DPD 16-30, Data 09/01/2026: 0 -> 1928
  ✓ FAIXA DPD 16-30, Data 12/01/2026: 0 -> 1928

Total de substituições: 24
Convertido de volta para formato longo: (552, 4)
Iniciando substituição...
Colunas de data identificadas: 6
Primeiras datas: ['08/01/2026', '09/01/2026', '12/01/2026']
  ✓ FAIXA DPD 16-30, Data 08/01/2026: 0 -> 1928
  ✓ FAIXA DPD 16-30, Data 09/01/2026: 0 -> 1928
  ✓ FAIXA DPD 16-30, Data 12/01/2026: 0 -> 1928

Total de substituições: 24
DataFrame atualizado:


,FAIXA,Indicador,08/01/2026,09/01/2026,12/01/2026,13/01/2026,14/01/2026,15/01/2026
14,DPD 16-30,Reachable_Portfolio,1928,1928,1928,1928,2782,2782
37,DPD 31-60,Reachable_Portfolio,2224,2224,2224,2224,2224,2224
60,DPD 61-90,Reachable_Portfolio,2094,2094,2094,2094,2094,2094
83,DPD 91-120,Reachable_Portfolio,1112,1112,1112,1112,1112,1112


In [47]:
df_transposto

,DATA,FAIXA,Indicador,Valor
0,08/01/2026,DPD 61-90,Assigned_Portfolio,2094
1,08/01/2026,DPD 91-120,Assigned_Portfolio,1112
2,08/01/2026,DPD 16-30,Assigned_Portfolio,2782
3,08/01/2026,DPD 31-60,Assigned_Portfolio,2224
4,09/01/2026,DPD 91-120,Assigned_Portfolio,1112
...,...,...,...,...
547,14/01/2026,DPD 91-120,Number_Of_Email_sent,0
548,15/01/2026,DPD 16-30,Number_Of_Email_sent,0
549,15/01/2026,DPD 91-120,Number_Of_Email_sent,0
550,15/01/2026,DPD 61-90,Number_Of_Email_sent,0


In [52]:
df_filtrado = df_transposto[df_transposto["Indicador"] == "Reachable_Portfolio"]
df_filtrado

,DATA,FAIXA,Indicador,Valor
24,08/01/2026,DPD 61-90,Reachable_Portfolio,0
25,08/01/2026,DPD 91-120,Reachable_Portfolio,0
26,08/01/2026,DPD 16-30,Reachable_Portfolio,0
27,08/01/2026,DPD 31-60,Reachable_Portfolio,0
28,09/01/2026,DPD 91-120,Reachable_Portfolio,0
29,09/01/2026,DPD 16-30,Reachable_Portfolio,0
30,09/01/2026,DPD 31-60,Reachable_Portfolio,0
31,09/01/2026,DPD 61-90,Reachable_Portfolio,0
32,12/01/2026,DPD 16-30,Reachable_Portfolio,0
33,12/01/2026,DPD 61-90,Reachable_Portfolio,0


In [53]:
import pandas as pd

# Exemplo: supondo que a coluna que guarda o nome do indicador se chama "Indicador"
df_filtrado = df_PAYJOY[df_PAYJOY["Indicador"] == "Reachable_Portfolio"]
df_filtrado

,FAIXA,Indicador,08/01/2026,09/01/2026,12/01/2026,13/01/2026,14/01/2026,15/01/2026
14,DPD 16-30,Reachable_Portfolio,1928,1928,1928,1928,2782,2782
37,DPD 31-60,Reachable_Portfolio,2224,2224,2224,2224,2224,2224
60,DPD 61-90,Reachable_Portfolio,2094,2094,2094,2094,2094,2094
83,DPD 91-120,Reachable_Portfolio,1112,1112,1112,1112,1112,1112
